In [ ]:

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

BASE = Path("")
INPUT_DIR = BASE / ""
OUTPUT_DIR = BASE / ""

FILES = {
    "email": "email_labeled.csv",
    "http": "http_labeled.csv",
    "file": "file_labeled.csv",
}

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
DIM = model.get_embedding_dimension()

for modality, filename in FILES.items():
    input_path = INPUT_DIR / filename
    modality_dir = OUTPUT_DIR / modality
    modality_dir.mkdir(parents=True, exist_ok=True)

    if not input_path.exists():
        raise FileNotFoundError(f"File not found: {input_path}")

    df = pd.read_csv(input_path, usecols=["user", "date", "content"], low_memory=False)

    if df.empty:
        raise ValueError(f"[{modality}] Empty dataset")

    if df["user"].isna().any():
        raise ValueError(f"[{modality}] Missing user values")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if df["date"].isna().any():
        raise ValueError(f"[{modality}] Invalid date values")

    df = df.sort_values(["user", "date"], kind="stable").reset_index(drop=True)

    for user, user_df in df.groupby("user", sort=False):
        texts = user_df["content"].fillna("").astype(str).str.strip()
        has_text = texts.ne("")

        embeddings = np.zeros((len(user_df), DIM), dtype=np.float32)

        if has_text.any():
            embeddings[has_text.to_numpy()] = model.encode(
                texts[has_text].tolist(),
                batch_size=BATCH_SIZE,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=False
            ).astype(np.float32)

        output_path = modality_dir / f"{modality}_{user}_embedding.pkl"

        with open(output_path, "wb") as f:
            pickle.dump(embeddings, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"[{modality.upper()}] Embeddings saved to {modality_dir}")
